# Plot from dicts

## Bring Plots from plotting_df_77_97_mrn

### Bring ***imports*** & ***det functions from*** plotting_df_77_97_mrn

#### ***imports*** 

In [1]:
# Imports required for Loading, sorting .csx files to create specific data sets ie mrn inbody readings. 
%run ./sys_funcs.py              # loads all the def functions in sys_funcs.py into memory
#import sys_funcs                 # gives access to these def function digitalform that are in memory
from pathlib import Path
import csv
import pandas as pd
import numpy as np
#import tkinter as tk
import pickle
from pathlib import Path
import csv
import os
import sys
import ipywidgets as widgets 
from datetime import datetime
from datetime import time
from sys_funcs import read_csv_to_array
from sys_funcs import clean_wsl_path
from sys_funcs import array_to_dt_row_dict
from sys_funcs import make_blnk_update_row_dict
from sys_funcs import transpose_csv_to_col_dict
#from sys_funcs import update_values_with_config, get_update_result
from sys_funcs import transfer_updates
from sys_funcs import get_dtv_range
from sys_funcs import universal_import
from sys_funcs import parse_inbody_timestamp
from sys_funcs import build_lut
from sys_funcs import extract_a_column_as_df
from sys_funcs import extract_multicolumns_as_df
from sys_funcs import validate_and_sort_timestamps
from sys_funcs import extract_and_filter_by_time_window
from sys_funcs import read_file_dual_path
from sys_funcs import write_file_dual_path
from sys_funcs import asc_to_csv_cnv
from collections.abc import Mapping
import re
import pandas as pd
from pathlib import Path
import pickle
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import unicodedata
import re
import pandas as pd
from pathlib import Path
import pickle
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import unicodedata
import re
#from sys_funcs import 

#### ***det functions***

In [19]:
def read_from_pkl(name):     # name is a string: read_from_pkl("name")import pickle and fill obj = name
    filename = f"{name}.pkl"
    print(filename)
    with open(filename, "rb") as f:
        loaded_obj = pickle.load(f)
    globals()[name] = loaded_obj
    print(f"{name} updated from {filename}")


In [39]:
def write_to_pkl(name):   # name is a string: write_to_pkl("name.pkl")
    import pickle
    import inspect
    
    # Access caller frame
    frame = inspect.currentframe().f_back

    # Pull the object from caller's local variables
    if name not in frame.f_locals:
        raise NameError(f"Variable '{name}' does not exist in caller scope")

    obj = frame.f_locals[name]

    # Write to pickle
    filename = f"{name}.pkl"
    with open(filename, "wb") as f:
        pickle.dump(obj, f)

    print(f"{name} written to {filename}")
    return None


In [27]:
def make_plt_lst_x(plt_list_nm, dct_i, key_i, dct_j, key_j):         # Make a new plt_lst_x
    return {
        0: plt_list_nm,
        1: dct_i,
        2: key_i,
        3: dct_j,
        4: key_j
    }


In [28]:
def add_row(plt_lst_dct, plt_lst_x):   # add the new row to "plt_lst_dct"
    next_idx = len(plt_lst_dct) + 1
    row_id = f"row_{next_idx:03d}"
    plt_lst_dct[row_id] = plt_lst_x
    return row_id


In [29]:
def verify_plt_lst_dct(plt_lst_dct):  # verify the contents of "plt_lst_dct"
    errors = []
    for row_id, row in plt_lst_dct.items():
        if not isinstance(row, dict):
            errors.append((row_id, "Row is not a dict"))
            continue
        for k in range(5):
            if k not in row:
                errors.append((row_id, f"Missing key {k}"))
    return errors


In [32]:
def delete_row_by_id(plt_lst_dct, row_id):  # delete row of "row_id" from "plt_lst_dct"
    if row_id in plt_lst_dct:
        del plt_lst_dct[row_id]
    return plt_lst_dct


In [31]:
def delete_row_by_name(plt_lst_dct, plt_list_nm):  # delete row of " plt_list_nm" from "plt_lst_dct"
    to_delete = [rid for rid, row in plt_lst_dct.items() if row[0] == plt_list_nm]
    for rid in to_delete:
        del plt_lst_dct[rid]
    return plt_lst_dct


In [35]:
def dispatch_by_name(plt_lst_dct, plt_list_nm):  # read row specified to give a "plt_lst_x"
    for row in plt_lst_dct.values():
        if row[0] == plt_list_nm:
            return row
    return None


In [33]:
def dispatch_by_row_id(plt_lst_dct, row_id):       # read row specified to give a "plt_lst_x"
    return plt_lst_dct.get(row_id, None)


In [36]:
def find_by_substring(plt_lst_dct, substring):  # find the row with a "plt_lst_nm" conatins the substring
    return {
        rid: row for rid, row in plt_lst_dct.items()
        if substring in row[0]
    }


In [37]:
def find_duplicates(plt_lst_dct):         # lst the duplicate rows so they can be fixed or deleded
    from collections import Counter
    
    names = [row[0] for row in plt_lst_dct.values()]
    dupes = {nm for nm, cnt in Counter(names).items() if cnt > 1}
    return {
        rid: row for rid, row in plt_lst_dct.items()
        if row[0] in dupes
    }


##### ***read & write from Local Storage***

In [43]:
 read_from_pkl("df_77_97_mrn")     # name is a string: read_from_pkl("ib_dct")import pickle


df_77_97_mrn.pkl
df_77_97_mrn updated from df_77_97_mrn.pkl


In [47]:
write_to_pkl("df_77_97_mrn")   # name is a string: write_to_pkl("ib_dct")

df_77_97_mrn written to df_77_97_mrn.pkl


In [7]:
# def read froms XL

In [8]:
# def write to XL

##### ***Slct_frm_dct***

##### ***add,edit,delete***

##### ***output***

In [15]:
# def plot_multi_dual(df, *cols, save_pdf=True):
def plot_multi_dual(df, *cols, save_pdf=True):
    # this uses a "df" and a "plotlist" and plots the graphs on screen, and optionally plots them into a PDF file for printing
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    import pandas as pd
    import os, re

    # Work on a safe copy
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # Color cycle (extend if needed)
    color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    fig, ax = plt.subplots(figsize=(12, 5))

    # First column on left axis
    ax.plot(
        df["timestamp"],
        df[cols[0]],
        marker="o",
        color=color_cycle[0],
        label=cols[0]
    )
    ax.set_ylabel(cols[0], color=color_cycle[0])
    ax.tick_params(axis="y", labelcolor=color_cycle[0])
    axes = [ax]

    # Additional columns on stacked right axes
    for i, col in enumerate(cols[1:], start=1):
        ax_new = ax.twinx()
        ax_new.spines.right.set_position(("axes", 1 + 0.1 * (i - 1)))

        color = color_cycle[i % len(color_cycle)]
        ax_new.plot(
            df["timestamp"],
            df[col],
            marker="o",
            color=color,
            label=col
        )
        ax_new.set_ylabel(col, color=color)
        ax_new.tick_params(axis="y", labelcolor=color)

        axes.append(ax_new)

    # Minor ticks every day
    ax.xaxis.set_minor_locator(mdates.DayLocator())

    # Major ticks weekly (Mondays)
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))

    # Rotate major tick labels for readability
    for label in ax.get_xticklabels(which="major"):
        label.set_rotation(45)
        label.set_ha("right")

    # Grid on major ticks
    ax.grid(which="major", linestyle="--", linewidth=0.7, alpha=0.7)

    plt.xticks(rotation=45)
    plt.title(" / ".join(cols) + " over time")
    plt.tight_layout()

    # === SAVE BLOCK (PDF, WSL-safe, sanitized filename) ===
    if save_pdf:
        save_dir = "/mnt/c/Users/bhuns/OneDrive/___ib_plots"
        os.makedirs(save_dir, exist_ok=True)

        # Build clean filename
        chart_title = " vs ".join(cols)
        chart_title = chart_title.replace("/", "_")
        chart_title = re.sub(r'[^A-Za-z0-9_-]+', '_', chart_title)

        # Add timestamp
        from datetime import datetime
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        chart_title = f"{timestamp}_{chart_title}"

        pdf_path = f"{save_dir}/{chart_title}.pdf"

        fig.savefig(pdf_path, dpi=300, bbox_inches="tight")

    plt.show()


#### ***Call_functions***

##### Import ***Dictionaries*** and  ***Data from df***

In [38]:
read_from_pkl("df_77_97_mrn")  # creats obj with name in ("")

df_77_97_mrn.pkl
df_77_97_mrn updated from df_77_97_mrn.pkl


In [45]:
write_to_pkl("df_77_97_mrn")    # name is a string: write_to_pkl("name.pkl"

df_77_97_mrn written to df_77_97_mrn.pkl


In [56]:
## plt_lst_dct ={}        # WARNING DELETE ALL CONTENTS OF "plt_lst_dct() "

In [61]:
# verify_plt_lst_dct(plt_lst_dct)

### ***source dat_cols*** to plot from ***df_77_97_mrn.columns.tolist***

In [64]:
df_77_97_mrn.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['timestamp', 'dtv', 'weight', 'vfa_(visceral_fat_area)', 'ecw/tbw',
       'ecw/tbw_of_left_leg_x', 'ecw/tbw_of_right_leg_x',
       'bmr_(basal_metabolic_rate)', 'smm_(skeletal_muscle_mass)',
       'khz-whole_body_phase_angle', 'whole_body_ecw/tbw_t_score',
       'ecw_(extracellular_water)', 'icw_(intracellular_water)',
       'ecw/tbw_of_left_leg_y', 'ecw/tbw_of_right_leg_y', 'ecw_of_left_leg',
       'ecw_of_right_leg', 'lower_limit_(ecw_of_left_leg_normal_range)',
       'lower_limit_(ecw_of_right_leg_normal_range)',
       'upper_limit_(ecw_of_left_leg_normal_range)',
       'upper_limit_(ecw_of_right_leg_normal_range)'],
      dtype='object')>

In [62]:
# make_plt_lst_x(plt_list_nm, dct_i, key_i, dct_j, key_j)

### Add a 1st row

#### fill in the info to male a plt_lst_x

In [72]:
plt_lst_x = make_plt_lst_x(            # filled in from dat_col list of  "df_77_97_mrn"
    "smmXvfa",
    "df_77_97_mrn",
    "smm_(skeletal_muscle_mass)",
    "df_77_97_mrn",
    "vfa_(visceral_fat_area)"
)


In [73]:
plt_lst_x

{0: 'smmXvfa',
 1: 'df_77_97_mrn',
 2: 'smm_(skeletal_muscle_mass)',
 3: 'df_77_97_mrn',
 4: 'vfa_(visceral_fat_area)'}

#### use ***plt_lst_x*** to populate ***plt_lst_dct***

In [74]:
add_row(plt_lst_dct, plt_lst_x)       # add new row "plt_lst_x" to "plt_lst_dct"

'row_001'

#### Verify new ***plt_lst_dct***

In [76]:
# verify 
plt_lst_dct

{'row_001': {0: 'smmXvfa',
  1: 'df_77_97_mrn',
  2: 'smm_(skeletal_muscle_mass)',
  3: 'df_77_97_mrn',
  4: 'vfa_(visceral_fat_area)'}}

### Add a 2nd row

#### fill in the info to male a plt_lst_x

In [78]:
plt_lst_x = make_plt_lst_x(            # filled in from dat_col list of  "df_77_97_mrn"
    "smmXphangl",
    "df_77_97_mrn",
    "smm_(skeletal_muscle_mass)",
    "df_77_97_mrn",
    'khz-whole_body_phase_angle'
)


#### use ***plt_lst_x*** to populate ***plt_lst_dct***

In [79]:
add_row(plt_lst_dct, plt_lst_x)       # add new row "plt_lst_x" to "plt_lst_dct"

'row_002'

#### Verify new ***plt_lst_dct***

In [80]:
# verify 
plt_lst_dct

{'row_001': {0: 'smmXvfa',
  1: 'df_77_97_mrn',
  2: 'smm_(skeletal_muscle_mass)',
  3: 'df_77_97_mrn',
  4: 'vfa_(visceral_fat_area)'},
 'row_002': {0: 'smmXphangl',
  1: 'df_77_97_mrn',
  2: 'smm_(skeletal_muscle_mass)',
  3: 'df_77_97_mrn',
  4: 'khz-whole_body_phase_angle'}}

### Use these to create a plot